In [ ]:
# 서울시 지하철역 엘리베이터 위치 정보

In [3]:
import requests               # 서울시 Open API 호출 (HTTP 요청용)
import time                   # API 호출 간격 조절 및 대기 시간 생성용
import polars as pl           # Pandas 대용, 고속 데이터프레임 처리 및 파케(Parquet) 저장용
from datetime import datetime # 수집 날짜/시간을 데이터에 기록하기 위한 타임스탬프용
from pathlib import Path      # 로컬 SSD 파일 경로(디렉토리 생성 및 관리) 제어용
import os
from sqlalchemy import create_engine, text
import sys
import tomllib



ModuleNotFoundError: No module named 'polars'

In [137]:
now = datetime.now()
dt_suffix = now.strftime("%Y%m%d")

## workspace/korea-public-data-pipeline-hub/seoul_subway/src/notebooks
current_dir = Path(".").resolve().as_posix()[3:]

CURRENT_NOTEBOOK_DIR = Path(os.getcwd())  # 현재 위치: .../src/notebooks
ROOT_DIR = CURRENT_NOTEBOOK_DIR.parents[2]                 # 최상위 지붕: .../korea-public-data-pipeline-hub
TARGET_SRC_DIR = ROOT_DIR / "seoul_subway" / "src"



D:\workspace\korea-public-data-pipeline-hub\01_seoul_subway_elevator\src\notebooks
D:\workspace\korea-public-data-pipeline-hub


In [130]:
# API_KEY = "7656714c4567757336356769446c6b" # KEY: 고유 인증키
# SERVICE_NAME = "tbTraficElvtr"
# START_INDEX = 1                            # START_INDEX: 페이징 시작 행 번호
# END_INDEX = 1000                          # END_INDEX: 페이징 종료 행 번호 |
# all_rows= []
#
# ##  http://openapi.seoul.go.kr:8088/(인증키)/xml/tbTraficElvtr/1/5/

In [139]:
# 🚀 2️⃣ [모듈 지도 세팅] 파이썬 임포트 기준점을 'src'로 완벽하게 고정!
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))


if str(TARGET_SRC_DIR) not in sys.path:
    sys.path.append(str(TARGET_SRC_DIR))
# 2️⃣ [홍 대리 맞춤형 팩트 경로] src 폴더가 기준이 되었으니 그 아래 'ods'에서 바로 가져오면 끝!
from ods.seoul_openapi_client import SeoulOpenApiClient

with open("../../config/dev.toml", "rb") as f:
    config = tomllib.load(f)

API_KEY = config["seoul_api"]["api_key"]
SERVICE_NAME = config["seoul_api"]["service_name"]
PAGE_SIZE = config["seoul_api"]["page_size"]


# 3. 🔄 루프 제어용 변수 선언 (메인에서만 숫자가 변합니다!)
all_rows = []
START_INDEX = 1
# END_INDEX = PAGE_SIZE  # 설정파일에서 가져온 1000으로 세팅!

# while True:

# 3️⃣ 인스턴스 생성 및 가동!
subway_api: SeoulOpenApiClient = SeoulOpenApiClient(API_KEY)
raw_data:dict  = subway_api.fetch_data(
    service_name=SERVICE_NAME,
    start_idx=START_INDEX,
    end_idx=PAGE_SIZE
)

# START_INDEX += PAGE_SIZE
# END_INDEX += PAGE_SIZE

NameError: name 'TARGET_SRC_DIR' is not defined

In [126]:
subway_status:dict = raw_data.get("tbTraficElvtr")

# 딕셔너리들이 담은 리스트로 변환)
subway_rows:list = subway_status.get("row")

AttributeError: 'NoneType' object has no attribute 'get'

In [127]:
len(subway_rows)

552

In [10]:
df: pl.DataFrame = pl.DataFrame(subway_rows)
print(type(df))

<class 'polars.dataframe.frame.DataFrame'>


In [ ]:
%%sql


In [11]:
df.write_parquet("../../data_lake/seoul_subway/seoul_subway_elevator.parquet")

In [119]:
df_read = pl.read_parquet("../../data_lake/seoul_subway/seoul_subway_elevator.parquet")
df_read.head(5)


NODE_TYPE,NODE_WKT,NODE_ID,NODE_TYPE_CD,SGG_CD,SGG_NM,EMD_CD,EMD_NM,SBWY_STN_CD,SBWY_STN_NM
str,str,f64,str,str,str,str,str,str,str
"""NODE""","""POINT(127.01744971746365 37.57…",212659.0,"""0""","""1111000000""","""종로구""","""1111017500""","""숭인동""","""268""","""동묘앞"""
"""NODE""","""POINT(127.01505874969273 37.57…",167351.0,"""1""","""1111000000""","""종로구""","""1111017400""","""창신동""","""270""","""창신"""
"""NODE""","""POINT(126.98515892357797 37.57…",211632.0,"""0""","""1111000000""","""종로구""","""1111013400""","""경운동""","""269""","""안국"""
"""NODE""","""POINT(127.01049296509703 37.57…",212410.0,"""0""","""1111000000""","""종로구""","""1111017400""","""창신동""","""272""","""동대문"""
"""NODE""","""POINT(127.02297735704515 37.57…",211950.0,"""0""","""1111000000""","""종로구""","""1111017500""","""숭인동""","""119""","""신설동"""


In [120]:
df_read.describe()

statistic,NODE_TYPE,NODE_WKT,NODE_ID,NODE_TYPE_CD,SGG_CD,SGG_NM,EMD_CD,EMD_NM,SBWY_STN_CD,SBWY_STN_NM
str,str,str,f64,str,str,str,str,str,str,str
"""count""","""552""","""552""",552.0,"""552""","""552""","""552""","""552""","""552""","""552""","""552"""
"""null_count""","""0""","""0""",0.0,"""0""","""0""","""0""","""0""","""0""","""0""","""0"""
"""mean""",null,null,176286.807971,null,null,null,null,null,null,null
"""std""",null,null,64274.422641,null,null,null,null,null,null,null
"""min""","""NODE""","""POINT(126.7980002435432 37.578…",408.0,"""0""","""1111000000""","""강남구""","""1111010700""","""가락동""","""1""","""4.19민주묘지"""
"""25%""",null,null,156068.0,null,null,null,null,null,null,null
"""50%""",null,null,212117.0,null,null,null,null,null,null,null
"""75%""",null,null,212972.0,null,null,null,null,null,null,null
"""max""","""NODE""","""POINT(127.17628284410385 37.55…",215561.0,"""2""","""1174000000""","""중랑구""","""1174011000""","""흥인동""","""99""","""흑석(중앙대입구)"""


In [117]:
df_ods_subway = df_read.select([
    # 1. 노드 타입 (일반 텍스트)
    pl.col("NODE_TYPE").cast(pl.String).alias("node_type"),
    # 2. GIS 공간 좌표 (WKT 텍스트 형식 유지)
    pl.col("NODE_WKT").cast(pl.String).alias("node_wkt"),
    # 3. 🌟 식별자 ID (정수형 변환)
    # 원본이 "212659.0" 형태의 문자열일 수 있으므로, Float로 먼저 읽고 Int64로 강제 변환하는 방어적 코딩!
    pl.col("NODE_ID").cast(pl.Float64).cast(pl.Int64).alias("node_id"),
    # 4. 타입 코드 (코드류는 무조건 앞자리 0 보존을 위해 String!)
    pl.col("NODE_TYPE_CD").cast(pl.String).alias("node_type_cd"),
    # 5. 시군구 코드 (코드류 -> String)
    pl.col("SGG_CD").cast(pl.String).alias("sgg_cd"),
    # 6. 시군구 명칭 (빈 값('')을 'Unknown'으로 방어)
    pl.when(pl.col("SGG_NM") == "")
      .then(pl.lit("Unknown"))
      .otherwise(pl.col("SGG_NM"))
      .cast(pl.String).alias("sgg_nm"),
    # 7. 읍면동 코드 (코드류 -> String)
    pl.col("EMD_CD").cast(pl.String).alias("emd_cd"),
    # 8. 읍면동 명칭 (일반 텍스트)
    pl.col("EMD_NM").cast(pl.String).alias("emd_nm"),
    # 9. 지하철역 코드 (코드류 -> String)
    pl.col("SBWY_STN_CD").cast(pl.String).alias("sbwy_stn_cd"),
    # 10. 지하철역 명칭 (일반 텍스트)
    pl.col("SBWY_STN_NM").cast(pl.String).alias("sbwy_stn_nm")
])
# 잘 변환되었는지 스키마(데이터 타입)와 결과물 확인!
print(df_ods_subway.schema)
print(df_ods_subway.head(5))

Schema({'node_type': String, 'node_wkt': String, 'node_id': Int64, 'node_type_cd': String, 'sgg_cd': String, 'sgg_nm': String, 'emd_cd': String, 'emd_nm': String, 'sbwy_stn_cd': String, 'sbwy_stn_nm': String})
shape: (5, 10)
┌───────────┬────────────┬─────────┬────────────┬───┬────────────┬────────┬────────────┬───────────┐
│ node_type ┆ node_wkt   ┆ node_id ┆ node_type_ ┆ … ┆ emd_cd     ┆ emd_nm ┆ sbwy_stn_c ┆ sbwy_stn_ │
│ ---       ┆ ---        ┆ ---     ┆ cd         ┆   ┆ ---        ┆ ---    ┆ d          ┆ nm        │
│ str       ┆ str        ┆ i64     ┆ ---        ┆   ┆ str        ┆ str    ┆ ---        ┆ ---       │
│           ┆            ┆         ┆ str        ┆   ┆            ┆        ┆ str        ┆ str       │
╞═══════════╪════════════╪═════════╪════════════╪═══╪════════════╪════════╪════════════╪═══════════╡
│ NODE      ┆ POINT(127. ┆ 212659  ┆ 0          ┆ … ┆ 1111017500 ┆ 숭인동 ┆ 268        ┆ 동묘앞    │
│           ┆ 0174497174 ┆         ┆            ┆   ┆            ┆        

SGG_NM,NODE_ID
u32,u32
0,0
